In [ ]:
# Cell 0 · Install & Import
!pip install awswrangler tensorflow matplotlib scikit-learn --quiet
import awswrangler as wr
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import warnings
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')
print('OK')

In [ ]:
# Cell 1 · Config
S3_BUCKET  = 'gold-lstm-forecast'
DATA_PATH  = f's3://{S3_BUCKET}/gold/xauusd_daily/features'
GOLD_PATH  = f's3://{S3_BUCKET}/gold/xauusd_daily/features/xauusd_features.parquet'
print(f'Data path : {DATA_PATH}')

In [ ]:
# Cell 2 · Load All Data
for fname in ['X_train.npy','X_test.npy','y_train.npy','y_test.npy',
              'feature_scaler.pkl','target_scaler.pkl',
              'wf_pred.npy','wf_actual.npy','test_dates.csv']:
    wr.s3.download(path=f'{DATA_PATH}/{fname}', local_file=f'/tmp/{fname}')

X_train = np.load('/tmp/X_train.npy')
X_test  = np.load('/tmp/X_test.npy')
y_train = np.load('/tmp/y_train.npy')
y_test  = np.load('/tmp/y_test.npy')

with open('/tmp/feature_scaler.pkl', 'rb') as f:
    feature_scaler = pickle.load(f)
with open('/tmp/target_scaler.pkl', 'rb') as f:
    target_scaler = pickle.load(f)

wf_pred   = np.load('/tmp/wf_pred.npy')
wf_actual = np.load('/tmp/wf_actual.npy')
test_dates = pd.read_csv('/tmp/test_dates.csv')['date'].values

y_actual = target_scaler.inverse_transform(y_test.reshape(-1,1)).ravel()

print('All loaded OK')
print(f'X_train : {X_train.shape} | X_test : {X_test.shape}')
print(f'wf_pred : {wf_pred.shape}')

In [ ]:
# Cell 3 · Baseline Models
FEATURES = ['close','return','ma7','ma14','ma30','ma60','volatility_7','momentum_7']

df = wr.s3.read_parquet(path=GOLD_PATH)
df = df.sort_values('date').reset_index(drop=True)

split_date = '2024-12-31'
train_df   = df[df['date'] <= split_date].copy()
test_df    = df[df['date'] >  split_date].copy()

# Naive Persistence
naive_pred   = test_df['close'].values
naive_actual = test_df['target'].values

# Linear Regression
X_tr_lr = feature_scaler.transform(train_df[FEATURES])
X_te_lr = feature_scaler.transform(test_df[FEATURES])
y_tr_lr = target_scaler.transform(train_df[['target']]).ravel()
y_te_lr = target_scaler.transform(test_df[['target']]).ravel()

lr      = LinearRegression().fit(X_tr_lr, y_tr_lr)
lr_pred = target_scaler.inverse_transform(lr.predict(X_te_lr).reshape(-1,1)).ravel()
lr_actual = target_scaler.inverse_transform(y_te_lr.reshape(-1,1)).ravel()

# AR(5)
close_all  = df['close'].values
target_all = df['target'].values
N = 5
# Extend the window upper bound by 1 (len(close_all)-N+i+1, not the naive
# len(close_all)-N+i) so the last row of Xar reaches all the way to the
# dataset's final target instead of stopping one short — otherwise AR(5)'s
# evaluation window ends one trading day earlier than every other model's.
Xar = np.column_stack([close_all[i:len(close_all)-N+i+1] for i in range(N)])
# yar[j] must be the close price on the day right after Xar[j]'s last input day
# (j+N-1). target_all[k] = close_all[k+1] (shift(-1)), so that's target_all[j+N-1],
# i.e. yar = target_all[N-1:] (now that Xar reaches the full dataset length).
yar = target_all[N-1:]
sp  = len(train_df) - N
ar  = LinearRegression().fit(Xar[:sp], yar[:sp])
# Prediction starts at sp+1, not sp: Xar[sp]'s last input day is the final
# training day, so yar[sp] is test_day_0's own close — the boundary value,
# not a forecast. Naive/LR/LSTM all start forecasting one day into the test
# period (test_day_1), so AR(5) has to start there too or its evaluation
# window is shifted a day earlier than everyone else's.
ar_pred   = ar.predict(Xar[sp+1:])
ar_actual = yar[sp+1:]

print('Baseline models done')

In [ ]:
# Cell 4 · Metrics
def get_metrics(actual, pred):
    mask = ~np.isnan(pred)
    a, p = np.array(actual)[mask], np.array(pred)[mask]
    return {
        'MAE' : round(mean_absolute_error(a, p), 2),
        'RMSE': round(np.sqrt(mean_squared_error(a, p)), 2),
        'R2'  : round(r2_score(a, p), 4),
        'MAPE': round(np.mean(np.abs((a-p)/a))*100, 3)
    }

def directional_accuracy(actual, pred):
    a          = np.array(actual)
    p          = np.array(pred)
    actual_dir = np.diff(a) > 0
    pred_dir   = np.diff(p) > 0
    return round(np.mean(actual_dir == pred_dir) * 100, 2)

min_len       = min(len(y_actual), len(lr_actual), len(naive_actual), len(ar_actual))
y_actual_trim = y_actual[-min_len:]
lr_trim       = lr_actual[-min_len:]
naive_trim    = naive_actual[-min_len:]
ar_trim       = ar_actual[-min_len:]
lr_pred_trim  = lr_pred[-min_len:]
naive_pd_trim = naive_pred[-min_len:]
ar_pred_trim  = ar_pred[-min_len:]

results = {
    'Naive Persistence': get_metrics(naive_trim,    naive_pd_trim),
    'AR(5) Baseline'   : get_metrics(ar_trim,       ar_pred_trim),
    'Linear Regression': get_metrics(lr_trim,       lr_pred_trim),
    'LSTM Walk-Forward': get_metrics(wf_actual,     wf_pred),
}

dir_acc = {
    'Naive Persistence': directional_accuracy(naive_trim,    naive_pd_trim),
    'AR(5) Baseline'   : directional_accuracy(ar_trim,       ar_pred_trim),
    'Linear Regression': directional_accuracy(lr_trim,       lr_pred_trim),
    'LSTM Walk-Forward': directional_accuracy(wf_actual,     wf_pred),
}

print('='*62)
print(f'{"Model":<24} {"MAE":>8} {"RMSE":>8} {"Dir Acc":>10}')
print('-'*62)
for model_name, m in results.items():
    da = dir_acc[model_name]
    print(f'{model_name:<24} {m["MAE"]:>8.2f} {m["RMSE"]:>8.2f} {da:>9.2f}%')
print('='*62)

In [ ]:
# Cell 5 · Actual vs Predicted Plot
fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle('LSTM Walk-Forward — Predicted vs Actual', fontsize=13, fontweight='bold')

dates_wf = pd.to_datetime(test_dates[-len(wf_pred):])

ax.plot(dates_wf, wf_actual, color='#E8A020', lw=1.5, label='Actual')
ax.plot(dates_wf, wf_pred,   color='#2ECC71', lw=1.2, label='LSTM Walk-Forward Predicted', linestyle='--')
ax.set_ylabel('Gold Price (USD)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('06_actual_vs_predicted.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved -> 06_actual_vs_predicted.png')

In [ ]:
# Cell 6 · Model Performance Comparison Chart
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Model Performance Comparison', fontsize=13, fontweight='bold')

model_names = list(results.keys())
short_names = ['Naive', 'AR(5)', 'Linear\nReg', 'LSTM\nWalk-Fwd']
colors      = ['#888888', '#EF9A9A', '#4A90D9', '#2ECC71']

# MAE
maes = [results[m]['MAE'] for m in model_names]
bars = axes[0].bar(short_names, maes, color=colors, width=0.55, edgecolor='white')
for bar, val in zip(bars, maes):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'${val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('MAE — Lower is Better')
axes[0].set_ylabel('USD')
axes[0].grid(True, alpha=0.3, axis='y')

# RMSE
rmses = [results[m]['RMSE'] for m in model_names]
bars2 = axes[1].bar(short_names, rmses, color=colors, width=0.55, edgecolor='white')
for bar, val in zip(bars2, rmses):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'${val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_title('RMSE — Lower is Better')
axes[1].set_ylabel('USD')
axes[1].grid(True, alpha=0.3, axis='y')

# Directional Accuracy
da_vals = [dir_acc[m] for m in model_names]
bars3   = axes[2].bar(short_names, da_vals, color=colors, width=0.55, edgecolor='white')
for bar, val in zip(bars3, da_vals):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[2].axhline(50, color='red', lw=1.5, linestyle='--', alpha=0.7)
axes[2].text(3.2, 51, '50%\nRandom', color='red', fontsize=8)
axes[2].set_title('Directional Accuracy — Higher is Better')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_ylim(0, 80)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('06_metrics_bar.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved -> 06_metrics_bar.png')

In [ ]:
# Cell 7 · Final Summary
wf_m = results['LSTM Walk-Forward']
wf_d = dir_acc['LSTM Walk-Forward']

print('=== Final Summary ===')
print(f'  LSTM Walk-Forward MAE  : ${wf_m["MAE"]}')
print(f'  LSTM Walk-Forward RMSE : ${wf_m["RMSE"]}')
print(f'  LSTM Walk-Forward MAPE : {wf_m["MAPE"]}%')
print(f'  LSTM Walk-Forward Dir Acc : {wf_d}%')
print()
print('  Known limitations:')
print('  - Model trained on prices up to $2700, test prices up to $5300')
print('  - Walk-forward retraining improves directional accuracy')
print('  - MAE/RMSE dominated by Naive due to persistent gold price surge')
print()
print('Notebook 06 DONE — Pipeline Complete')